In [ ]:
import os
import time
import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from openpyxl.styles import Alignment



# =========================
# Chrome配置（和B站一致）
# =========================

options = Options()


options.add_argument(
    r"--user-data-dir=C:\Users\12082\AppData\Local\Google\Chrome\SeleniumData"
)


options.add_argument(
    "--start-maximized"
)



driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    ),
    options=options
)


print("ChromeDriver加载成功！")



# =========================
# 打开公众号后台
# =========================

url = "https://mp.weixin.qq.com/"


driver.get(url)



wait = WebDriverWait(driver, 300)



# =========================
# 登录等待
# =========================

try:

    print("请登录微信公众号...")


    wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//span[text()='内容管理']"
            )
        )
    )


    print("登录成功！")


except Exception as e:

    print(
        "登录失败:",
        e
    )

    driver.quit()

    exit()



# =========================
# 进入发表记录
# =========================

try:


    driver.find_element(
        By.XPATH,
        "//span[text()='内容管理']"
    ).click()



    wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//span[text()='发表记录']"
            )
        )
    ).click()



    print(
        "进入发表记录页面"
    )


except Exception as e:

    print(
        "进入页面失败:",
        e
    )

    driver.quit()

    exit()



# =========================
# 数据提取函数
# =========================

def clean_number(text):

    if not text:
        return "0"


    return (
        text
        .replace(",","")
        .strip()
    )



def extract_page_data():


    page_data = []


    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )


    articles = soup.find_all(
        "div",
        class_="weui-desktop-mass-appmsg__bd"
    )


    for article in articles:


        try:


            title_tag = article.find(
                "a",
                class_="weui-desktop-mass-appmsg__title"
            )


            title = (
                title_tag.find("span").text.strip()
                if title_tag
                else "N/A"
            )



            def get_data(class_name):

                item = article.find(
                    "div",
                    class_=class_name
                )


                if item:

                    span = item.find(
                        "span",
                        class_="weui-desktop-mass-media__data__inner"
                    )

                    if span:

                        return clean_number(
                            span.text
                        )


                return "0"



            read_count = get_data(
                "weui-desktop-mass-media__data appmsg-view"
            )


            likes = get_data(
                "weui-desktop-mass-media__data appmsg-like"
            )


            shares = get_data(
                "weui-desktop-mass-media__data appmsg-share"
            )


            on_view = get_data(
                "weui-desktop-mass-media__data appmsg-haokan"
            )


            comments = get_data(
                "weui-desktop-mass-media__data appmsg-comment"
            )



            page_data.append(
                {
                    "标题":title,
                    "阅读人数":read_count,
                    "点赞人数":likes,
                    "分享人数":shares,
                    "在看人数":on_view,
                    "留言条数":comments
                }
            )


        except Exception as e:

            print(
                "文章解析失败:",
                e
            )


    print(
        f"本页提取 {len(page_data)} 篇文章"
    )


    return page_data




# =========================
# 自动翻页
# =========================

all_articles_data=[]

page=1



while len(all_articles_data)<50:


    print(
        f"正在提取第{page}页"
    )


    current_data = extract_page_data()


    all_articles_data.extend(
        current_data
    )



    if len(all_articles_data)>=50:

        break



    try:


        next_button = wait.until(
            EC.element_to_be_clickable(
                (
                    By.LINK_TEXT,
                    "下一页"
                )
            )
        )


        next_button.click()


        page+=1


        print(
            "点击下一页"
        )


        time.sleep(2)



    except Exception as e:


        print(
            "没有下一页:",
            e
        )

        break



# 关闭浏览器

driver.quit()



# =========================
# 保存Excel
# =========================

all_articles_data = all_articles_data[:50]


if all_articles_data:


    df=pd.DataFrame(
        all_articles_data
    )


    output_file=os.path.join(
        os.path.expanduser("~"),
        "Desktop",
        "微信文章数据.xlsx"
    )



    df.to_excel(
        output_file,
        index=False,
        sheet_name="微信文章数据"
    )



    print(
        f"保存成功:{output_file}"
    )



    os.startfile(
        output_file
    )


else:


    print(
        "没有提取到数据"
    )

ChromeDriver加载成功！
请登录微信公众号...
登录失败: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=150.0.7871.114)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7c5cca865+152a5]
	chromedriver!GetHandleVerifier [0x7ff7c5cca8c0+15300]
	chromedriver!(No symbol) [0x7ff7c582596d]
	chromedriver!(No symbol) [0x7ff7c57fc782]
	chromedriver!(No symbol) [0x7ff7c58b01d6]
	chromedriver!(No symbol) [0x7ff7c58cd5b2]
	chromedriver!(No symbol) [0x7ff7c5872b3c]
	chromedriver!(No symbol) [0x7ff7c5873a53]
	chromedriver!GetHandleVerifier [0x7ff7c62ad3a1+5f7de1]
	chromedriver!GetHandleVerifier [0x7ff7c62a7a3b+5f247b]
	chromedriver!GetHandleVerifier [0x7ff7c62cbca5+6166e5]
	chromedriver!GetHandleVerifier [0x7ff7c5ce72ae+31cee]
	chromedriver!GetHandleVerifier [0x7ff7c5cefe3c+3a87c]
	chromedriver!GetHandleVerifier [0x7ff7c5cd4404+1ee44]
	chromedriver!GetHandleVerifier [0x7ff7c5cd4594+1efd4]
	chromedriver!GetHandleVerifier [0x7ff7c5cb75c7+2007]
	KERNEL32